# MedViHall — Dataset Download Pipeline
### Pilot: 200 images per dataset | Full: 2,000 images per modality

**Datasets covered:**
| # | Dataset | Modality | Source | Auth |
|---|---|---|---|---|
| 1 | CheXpert v1.0-512 | Radiology | HuggingFace (StanfordAIMI) | None |
| 2 | ISIC 2020 | Dermatology | ISIC Public API | None |
| 3 | HAM10000 | Dermatology | Kaggle | Kaggle key |
| 4 | PatchCamelyon (PCam) | Histopathology | HuggingFace | None |
| 5 | NCT-CRC-HE-100K | Histopathology | HuggingFace | None |
| 6 | REFUGE | Ophthalmology | Kaggle | Kaggle key |
| 7 | ORIGA-light | Ophthalmology | Kaggle | Kaggle key |

**Run order:** Execute sections top to bottom. Each section is independent.  
**After all downloads:** Run the final *Merge & Validate* section to produce clean ground truth files.


---
## Section 0 — Installation & Global Config
Install all required libraries and define shared settings used by every download section.


In [27]:
# KGAT_f03c36e6e5b731ac0a5d8cb6430e17b4

In [28]:
# ── Install dependencies ──────────────────────────────────────────────────
%pip install datasets==2.20.0 huggingface_hub==0.24.0 Pillow==10.4.0 \
            pandas==2.2.2 numpy==1.26.4 requests==2.32.3 tqdm==4.66.5 \
            kaggle==1.6.14 scikit-learn==1.5.1 -q

print("✓ All packages installed")


Note: you may need to restart the kernel to use updated packages.
✓ All packages installed



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
# ── Global config ─────────────────────────────────────────────────────────
import os, json, random, time, subprocess, shutil, zipfile
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import requests

# ── Pilot vs Full ──────────────────────────────────────────────────────────
# Set MODE = "pilot"  → 200 images per dataset (~1,400 total, fast)
# Set MODE = "full"   → 2,000 images per modality (~8,000 total)
MODE = "full"  # ← Change to "full" for complete evaluation

PILOT_N = {
    "chexpert":  {"S1": 50,  "S2": 50,  "S3": 50,  "S4": 50},
    "isic":      {"DS1": 70, "DS2": 30, "DS3": 100},
    "ham":       {"DS1": 50, "DS2": 50, "DS3": 100},
    "pcam":      {"HS1": 100,"HS2": 0,  "HS3": 100},
    "nct":       {"HS1": 50, "HS2": 20, "HS3": 50},
    "refuge":    {"OS1": 70, "OS2": 60, "OS3": 70},
    "origa":     {"OS1": 80, "OS2": 70, "OS3": 50},
}
FULL_N = {
    "chexpert":  {"S1": 500, "S2": 500, "S3": 500, "S4": 500},
    "isic":      {"DS1": 1333,"DS2": 83,"DS3": 584},
    "ham":       {"DS1": 300,"DS2": 300,"DS3": 400},
    "pcam":      {"HS1": 700,"HS2": 0,  "HS3": 700},
    "nct":       {"HS1": 400,"HS2": 200,"HS3": 400},
    "refuge":    {"OS1": 700,"OS2": 600,"OS3": 700},
    "origa":     {"OS1": 300,"OS2": 200,"OS3": 150},
}
N = PILOT_N if MODE == "pilot" else FULL_N

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# ── Directory structure ────────────────────────────────────────────────────
BASE_DIR  = Path("medvihall_data")
RAW_DIR   = BASE_DIR / "data" / "raw"
GT_DIR    = BASE_DIR / "data" / "ground_truth"
TMP_DIR   = BASE_DIR / "data" / "tmp"

for d in [
    RAW_DIR / "radiology"     / "chexpert",
    RAW_DIR / "dermatology"   / "isic2020",
    RAW_DIR / "dermatology"   / "ham10000",
    RAW_DIR / "histopathology"/ "pcam",
    RAW_DIR / "histopathology"/ "nct_crc",
    RAW_DIR / "ophthalmology" / "refuge",
    RAW_DIR / "ophthalmology" / "origa",
    GT_DIR, TMP_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Mode: {MODE.upper()}")
print(f"✓ Directory structure created under: {BASE_DIR.resolve()}")
print(f"✓ Sample targets:")
for ds, strata in N.items():
    print(f"   {ds:<12} {strata}  →  total={sum(strata.values())}")


✓ Mode: FULL
✓ Directory structure created under: C:\Users\PC\Desktop\Programs\MediViHall\medvihall_data
✓ Sample targets:
   chexpert     {'S1': 500, 'S2': 500, 'S3': 500, 'S4': 500}  →  total=2000
   isic         {'DS1': 1333, 'DS2': 83, 'DS3': 584}  →  total=2000
   ham          {'DS1': 300, 'DS2': 300, 'DS3': 400}  →  total=1000
   pcam         {'HS1': 700, 'HS2': 0, 'HS3': 700}  →  total=1400
   nct          {'HS1': 400, 'HS2': 200, 'HS3': 400}  →  total=1000
   refuge       {'OS1': 700, 'OS2': 600, 'OS3': 700}  →  total=2000
   origa        {'OS1': 300, 'OS2': 200, 'OS3': 150}  →  total=650


---
## Section 0b — Kaggle API Key Setup
Required for: HAM10000, REFUGE, ORIGA.  
**Skip this section if you already have `~/.kaggle/kaggle.json`.**


In [30]:
# KGAT_231304a703e1a263afe8bfc926f894f8

In [31]:
# ── Kaggle API setup (correct format) ─────────────────────────────────────
import os, json
from pathlib import Path

# ── Fill these two values ──────────────────────────────────────────────────
KAGGLE_USERNAME = "Anuruddhapaul44"          # ← your Kaggle username
KAGGLE_KEY      = "KGAT_231304a703e1a263afe8bfc926f894f8" # ← your API key

# ── Write kaggle.json in the format the CLI expects ────────────────────────
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

cfg_path = kaggle_dir / "kaggle.json"
cfg_path.write_text(json.dumps({
    "username": KAGGLE_USERNAME,
    "key":      KAGGLE_KEY
}))
os.chmod(cfg_path, 0o600)

# Verify the file looks right
written = json.loads(cfg_path.read_text())
print(f"✓ Written to {cfg_path}")
print(f"  username: {written['username']}")
print(f"  key:      {written['key'][:8]}{'*' * (len(written['key'])-8)}")

# ── Test authentication ────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ["kaggle", "datasets", "list", "--max-size", "1"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("\n✓ Kaggle authenticated successfully")
else:
    print(f"\n✗ Still failing:\n{result.stderr.strip()}")

✓ Written to C:\Users\PC\.kaggle\kaggle.json
  username: Anuruddhapaul44
  key:      KGAT_231*****************************

✓ Kaggle authenticated successfully


In [32]:
# ── Verify Kaggle works ────────────────────────────────────────────────────
result = subprocess.run(["kaggle", "datasets", "list", "--max-size", "1"],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Kaggle API is working")
else:
    print("✗ Kaggle API error:")
    print(result.stderr)
    print("\nFix: ensure ~/.kaggle/kaggle.json exists with valid credentials")


✓ Kaggle API is working


---
## Section 1 — Radiology: CheXpert
**Source:** `StanfordAIMI/CheXpert-v1.0-512` (HuggingFace — official Stanford AIMI release)  
**Auth:** None required  
**Size:** ~11 GB full dataset (cached after first download)  
**Images:** 512×512 frontal chest X-rays with CheXpert 14-label annotations  

**Stratification:**
- `S1` — Normal (No Finding == 1, all pathologies negative)  
- `S2` — Single finding (exactly 1 pathology label positive)  
- `S3` — Multi finding (2–3 positive labels)  
- `S4` — Complex (4+ positive labels)


In [33]:
# ── CheXpert label definitions ─────────────────────────────────────────────
PATHOLOGY_LABELS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion",
    "Lung Opacity", "Pleural Effusion", "Pleural Other",
    "Pneumonia", "Pneumothorax", "Support Devices"
]

def chexpert_stratum(sample):
    if sample.get("No Finding", 0) == 1.0:
        return "S1"
    n = sum(1 for l in PATHOLOGY_LABELS if sample.get(l, -1) == 1.0)
    if n == 1:   return "S2"
    elif n <= 3: return "S3"
    else:        return "S4"

print("✓ CheXpert stratum function defined")


✓ CheXpert stratum function defined


In [34]:
# hf_REDACTED

In [35]:
# # ═══════════════════════════════════════════════════════════════════════════
# # SECTION 1 — RADIOLOGY: CheXpert
# # Strategy : kagglehub downloads full dataset → we copy 2,500 stratified
# #            images (625 per stratum) → you hand-pick best 2,000 → delete rest
# # ═══════════════════════════════════════════════════════════════════════════

# import json, shutil
# from pathlib import Path
# from PIL import Image
# import pandas as pd
# import kagglehub
# from tqdm.notebook import tqdm

# CHEXPERT_DIR  = RAW_DIR / "radiology" / "chexpert"
# CHEXPERT_DIR.mkdir(parents=True, exist_ok=True)

# PATHOLOGY_LABELS = [
#     "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
#     "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion",
#     "Lung Opacity", "Pleural Effusion", "Pleural Other",
#     "Pneumonia", "Pneumothorax", "Support Devices"
# ]

# def chexpert_stratum(row) -> str:
#     if row.get("No Finding", 0) == 1.0:
#         return "S1"
#     n = sum(1 for l in PATHOLOGY_LABELS if row.get(l, -1) == 1.0)
#     if n == 1:   return "S2"
#     elif n <= 3: return "S3"
#     else:        return "S4"

# # 625 per stratum = 2,500 total (500 target + 125 buffer each)
# TARGETS_WITH_BUFFER = {"S1": 500, "S2": 500, "S3": 500, "S4": 500}

# # ── Step 1: Download full dataset via kagglehub ────────────────────────────
# print("Step 1/4 — Downloading CheXpert via kagglehub ...")
# print("  ☕ First run downloads full dataset — subsequent runs use cache\n")

# dataset_path = "C:/Users/PC/.cache/kagglehub/datasets/ashery/chexpert"
# print(f"\n✓ Dataset path: {dataset_path}")

# # Show what landed
# # print("\n  Top-level contents:")
# # for p in sorted(dataset_path.iterdir())[:15]:
# #     size = f"{p.stat().st_size/1e6:.1f} MB" if p.is_file() else "dir"
# #     print(f"    {p.name}  ({size})")

# # ── Step 2: Find train.csv ─────────────────────────────────────────────────
# print("\nStep 2/4 — Loading train.csv ...")

# csv_candidates = list(dataset_path.rglob("train.csv"))
# print(f"  Found: {csv_candidates}")

# assert csv_candidates, (
#     f"train.csv not found under {dataset_path}\n"
#     f"Files present: {[f.name for f in dataset_path.rglob('*') if f.is_file()][:20]}"
# )

# csv_path = csv_candidates[0]
# df = pd.read_csv(csv_path)
# print(f"✓ Loaded {len(df):,} records")
# print(f"  Columns: {list(df.columns)}")
# print(df.head(2).to_string())

# # ── Step 3: Stratify + select 2,500 images ────────────────────────────────
# print("\nStep 3/4 — Selecting 2,500 images (625 per stratum) ...")

# # Keep frontal views only
# df = df[df["Path"].str.contains("frontal", case=False, na=False)].copy()
# print(f"  Frontal views available: {len(df):,}")

# df["stratum"] = df.apply(chexpert_stratum, axis=1)

# print(f"\n  Available per stratum:")
# for s, n in TARGETS_WITH_BUFFER.items():
#     available = len(df[df["stratum"] == s])
#     print(f"    {s}: {available:>7,} available  →  {n} requested")

# selected = pd.concat([
#     df[df["stratum"] == s].sample(
#         n=min(n, len(df[df["stratum"] == s])),
#         random_state=RANDOM_SEED
#     )
#     for s, n in TARGETS_WITH_BUFFER.items()
# ]).reset_index(drop=True)

# print(f"\n✓ Selected {len(selected)} images")
# print(f"  Strata breakdown: {dict(selected['stratum'].value_counts())}")

# # ── Step 4: Copy + save as JPEG + build ground truth ──────────────────────
# print(f"\nStep 4/4 — Copying {len(selected)} images to working directory ...")
# print(f"  Destination: {CHEXPERT_DIR}\n")

# gt_chexpert    = []
# saved_chexpert = 0
# failed         = []

# for _, row in tqdm(selected.iterrows(),
#                    total=len(selected), desc="Copying CheXpert"):

#     # Build full source path — try multiple path styles
#     rel_path  = str(row["Path"]).replace("\\", "/")
#     src_path  = dataset_path / rel_path

#     # Fallback: search by filename if direct path fails
#     if not src_path.exists():
#         fname   = Path(rel_path).name
#         matches = list(dataset_path.rglob(fname))
#         src_path = matches[0] if matches else None

#     if src_path is None or not Path(src_path).exists():
#         failed.append(rel_path)
#         continue

#     stratum = row["stratum"]
#     img_id  = f"chexpert_{stratum}_{saved_chexpert:04d}"
#     fname   = f"{img_id}.jpg"
#     fpath   = CHEXPERT_DIR / fname

#     # Skip if already saved (resume-safe)
#     if fpath.exists():
#         saved_chexpert += 1
#         continue

#     # Convert to RGB JPEG
#     try:
#         img = Image.open(src_path).convert("RGB")
#         img.save(str(fpath), "JPEG", quality=95)
#     except Exception as e:
#         failed.append(f"{rel_path} ({e})")
#         continue

#     pos = [l for l in PATHOLOGY_LABELS if row.get(l, -1) == 1.0]
#     neg = [l for l in PATHOLOGY_LABELS if row.get(l,  0) == 0.0]

#     gt_chexpert.append({
#         "image_id":        img_id,
#         "filename":        fname,
#         "source":          "CheXpert-v1.0",
#         "modality":        "radiology",
#         "stratum":         stratum,
#         "view":            "frontal",
#         "positive_labels": pos,
#         "negative_labels": neg,
#         "no_finding":      bool(row.get("No Finding", 0) == 1.0),
#         "n_positive":      len(pos),
#         "patient_sex":     str(row.get("Sex", "")),
#         "patient_age":     row.get("Age", None),
#         "original_path":   rel_path,
#         "keep":            True,   # ← set to False for images to delete later
#     })
#     saved_chexpert += 1

# # Save ground truth
# gt_path = GT_DIR / "radiology_gt.json"
# with open(gt_path, "w") as f:
#     json.dump(gt_chexpert, f, indent=2)

# # ── Summary ────────────────────────────────────────────────────────────────
# strata_counts = {}
# for r in gt_chexpert:
#     strata_counts[r["stratum"]] = strata_counts.get(r["stratum"], 0) + 1

# print(f"\n{'='*58}")
# print(f"✓ CheXpert — complete")
# print(f"  Saved     : {saved_chexpert} / {len(selected)} images")
# print(f"  GT file   : {gt_path}")
# print(f"  Strata    : {strata_counts}")
# if failed:
#     print(f"  ⚠ Failed  : {len(failed)} — {failed[:3]}")
# print(f"\n  NEXT STEP:")
# print(f"  → Open {CHEXPERT_DIR} in File Explorer")
# print(f"  → Delete any bad/corrupt/unclear images")
# print(f"  → Run the pruning cell below to sync GT JSON with remaining files")
# print(f"{'='*58}")

---
## Section 2 — Dermatology: ISIC 2020
**Source:** ISIC Archive Public API (`https://api.isic-archive.com/api/v2/`)  
**Auth:** None required  
**Images:** Dermoscopy images with malignancy labels  

**Stratification:**
- `DS1` — Benign: nevus, dermatofibroma, seborrheic keratosis, vascular
- `DS2` — Borderline: actinic keratosis, lentigo
- `DS3` — Malignant: melanoma, basal cell carcinoma


In [36]:
# ── ISIC stratum mapping ───────────────────────────────────────────────────
ISIC_STRATUM = {
    "nevus": "DS1", "dermatofibroma": "DS1", "seborrheic_keratosis": "DS1",
    "vascular_lesion": "DS1", "pigmented_benign_keratosis": "DS1",
    "cafe_au_lait_macule": "DS1",
    "actinic_keratosis": "DS2", "squamous_cell_carcinoma": "DS2",
    "lentigo_nos": "DS2", "lichenoid_keratosis": "DS2",
    "melanoma": "DS3", "basal_cell_carcinoma": "DS3",
}
ISIC_DIR = RAW_DIR / "dermatology" / "isic2020"
targets_isic = N["isic"]
print(f"ISIC 2020 targets: {targets_isic}")


ISIC 2020 targets: {'DS1': 1333, 'DS2': 83, 'DS3': 584}


In [37]:
# # ═══════════════════════════════════════════════════════════════════════════
# # SECTION 2 — DERMATOLOGY: ISIC 2020
# # Strategy : kagglehub downloads full dataset → select 2,500 stratified
# #            images (625 per stratum, 500 target + 125 buffer)
# #            → you hand-pick 2,000 → run pruning cell to sync GT
# # Dataset  : kaggle: cdeotte/jpeg-isic2020-256x256
# # ═══════════════════════════════════════════════════════════════════════════

# import json, shutil
# from pathlib import Path
# from PIL import Image
# import pandas as pd
# import kagglehub
# from tqdm.notebook import tqdm

# ISIC_DIR = RAW_DIR / "dermatology" / "isic2020"
# ISIC_DIR.mkdir(parents=True, exist_ok=True)

# # Diagnosis → MedViHall stratum
# ISIC_STRATUM_MAP = {
#     # Benign (DS1)
#     "nevus":                      "DS1",
#     "melanocytic_nevus":          "DS1",
#     "dermatofibroma":             "DS1",
#     "seborrheic_keratosis":       "DS1",
#     "vascular_lesion":            "DS1",
#     "pigmented_benign_keratosis": "DS1",
#     "cafe_au_lait_macule":        "DS1",
#     "lentigo_simplex":            "DS1",
#     # Borderline (DS2)
#     "actinic_keratosis":          "DS2",
#     "squamous_cell_carcinoma":    "DS2",
#     "lentigo_nos":                "DS2",
#     "lichenoid_keratosis":        "DS2",
#     # Malignant (DS3)
#     "melanoma":                   "DS3",
#     "basal_cell_carcinoma":       "DS3",
# }

# def isic_stratum(row) -> str:
#     # Primary: use diagnosis column if available
#     diag = str(row.get("diagnosis", "") or "").lower().strip()
#     diag = diag.replace(" ", "_").replace("-", "_")
#     if diag in ISIC_STRATUM_MAP:
#         return ISIC_STRATUM_MAP[diag]

#     # Secondary: use target column (0=benign, 1=malignant)
#     target = row.get("target", row.get("benign_malignant", None))
#     if target == 1 or str(target).lower() == "malignant":
#         return "DS3"
#     if target == 0 or str(target).lower() == "benign":
#         return "DS1"

#     return None   # cannot classify

# TARGETS_WITH_BUFFER = {"DS1": 1333, "DS2": 250, "DS3": 875}
# # DS2 (borderline) is rare in ISIC 2020 — keep buffer smaller
# # Total = 2,000 with 125+ buffer in DS1/DS3

# # ── Step 1: Download via kagglehub ─────────────────────────────────────────
# print("Step 1/4 — Downloading ISIC 2020 via kagglehub ...")
# print("  Dataset: cdeotte/jpeg-isic2020-256x256")
# print("  ☕ First run downloads full dataset (~2.5 GB)\n")

# dataset_path = Path(kagglehub.dataset_download("ziadloo/isic-2020-256x256"))
# print(f"\n✓ Dataset path: {dataset_path}")

# print("\n  Contents:")
# for p in sorted(dataset_path.iterdir()):
#     if p.is_dir():
#         n_files = len(list(p.glob("*")))
#         print(f"    📁 {p.name}/  ({n_files} files)")
#     else:
#         print(f"    📄 {p.name}  ({p.stat().st_size/1e6:.1f} MB)")

# # ── Step 2: Load labels CSV ────────────────────────────────────────────────
# print("\nStep 2/4 — Loading labels ...")

# # Try common CSV names
# csv_candidates = (
#     list(dataset_path.rglob("train.csv")) +
#     list(dataset_path.rglob("train_labels.csv")) +
#     list(dataset_path.rglob("*.csv"))
# )
# print(f"  CSVs found: {[c.name for c in csv_candidates]}")

# # Use train.csv preferentially
# csv_path = next((c for c in csv_candidates if "train" in c.name.lower()),
#                 csv_candidates[0] if csv_candidates else None)
# assert csv_path, "No CSV found — check dataset contents above"

# df = pd.read_csv(csv_path)
# print(f"✓ Loaded {len(df):,} records from {csv_path.name}")
# print(f"  Columns: {list(df.columns)}")
# print(df.head(3).to_string())

# # ── Step 3: Assign strata + select 2,000 images ───────────────────────────
# print("\nStep 3/4 — Stratifying ...")

# df["stratum"] = df.apply(isic_stratum, axis=1)
# df_valid = df.dropna(subset=["stratum"]).copy()

# print(f"  Classifiable: {len(df_valid):,} / {len(df):,}")
# print(f"  Stratum distribution:")
# for s, n in TARGETS_WITH_BUFFER.items():
#     available = len(df_valid[df_valid["stratum"] == s])
#     print(f"    {s}: {available:>7,} available  →  {n} requested")

# selected = pd.concat([
#     df_valid[df_valid["stratum"] == s].sample(
#         n=min(n, len(df_valid[df_valid["stratum"] == s])),
#         random_state=RANDOM_SEED
#     )
#     for s, n in TARGETS_WITH_BUFFER.items()
#     if len(df_valid[df_valid["stratum"] == s]) > 0
# ]).reset_index(drop=True)

# print(f"\n✓ Selected {len(selected)} images")
# print(f"  {dict(selected['stratum'].value_counts())}")

# # ── Step 4: Find image files + figure out folder structure ─────────────────
# print("\nStep 4/4 — Copying images + building ground truth ...")

# # Find all jpg folders
# img_dirs = [d for d in dataset_path.rglob("*") if d.is_dir()]
# print(f"  Image directories: {[d.name for d in img_dirs[:10]]}")

# # Build filename → full path lookup (faster than rglob per image)
# print("  Building image path index ...")
# img_index = {}
# for img_path in dataset_path.rglob("*.jpg"):
#     img_index[img_path.stem] = img_path   # stem = filename without extension
# for img_path in dataset_path.rglob("*.jpeg"):
#     img_index[img_path.stem] = img_path

# print(f"  Indexed {len(img_index):,} images")

# # ── Copy + save ────────────────────────────────────────────────────────────
# gt_isic    = []
# saved_isic = 0
# failed     = []

# # Detect image ID column name
# id_col = next((c for c in df.columns
#                if any(x in c.lower() for x in ["image_name","image_id","isic_id","name"])),
#               df.columns[0])
# print(f"  Using '{id_col}' as image ID column\n")

# for _, row in tqdm(selected.iterrows(),
#                    total=len(selected), desc="Copying ISIC 2020"):

#     img_id_orig = str(row[id_col]).replace(".jpg", "").strip()
#     src_path    = img_index.get(img_id_orig)

#     if src_path is None:
#         failed.append(img_id_orig)
#         continue

#     stratum = row["stratum"]
#     img_id  = f"isic2020_{stratum}_{saved_isic:04d}"
#     fname   = f"{img_id}.jpg"
#     fpath   = ISIC_DIR / fname

#     if fpath.exists():
#         saved_isic += 1
#         continue

#     try:
#         img = Image.open(src_path).convert("RGB")
#         img.save(str(fpath), "JPEG", quality=95)
#     except Exception as e:
#         failed.append(f"{img_id_orig} ({e})")
#         continue

#     gt_isic.append({
#         "image_id":        img_id,
#         "filename":        fname,
#         "source":          "ISIC2020",
#         "modality":        "dermatology",
#         "stratum":         stratum,
#         "original_id":     img_id_orig,
#         "diagnosis":       str(row.get("diagnosis", "")),
#         "target":          int(row.get("target", -1)),
#         "malignant":       stratum == "DS3",
#         "sex":             str(row.get("sex", "")),
#         "age_approx":      row.get("age_approx", None),
#         "anatomical_site": str(row.get("anatom_site_general", "")),
#         "keep":            True,
#     })
#     saved_isic += 1

# # Save GT
# gt_path = GT_DIR / "dermatology_isic2020_gt.json"
# with open(gt_path, "w") as f:
#     json.dump(gt_isic, f, indent=2)

# strata_counts = {}
# for r in gt_isic:
#     strata_counts[r["stratum"]] = strata_counts.get(r["stratum"], 0) + 1

# print(f"\n{'='*55}")
# print(f"✓ ISIC 2020 — complete")
# print(f"  Saved   : {saved_isic} / {len(selected)} images")
# print(f"  GT file : {gt_path}")
# print(f"  Strata  : {strata_counts}")
# if failed:
#     print(f"  ⚠ Failed: {len(failed)} — {failed[:3]}")
# print(f"\n  NEXT STEP:")
# print(f"  → Open {ISIC_DIR} in File Explorer")
# print(f"  → Delete any bad images (blurry, corrupt, wrong content)")
# print(f"  → Run pruning cell below to sync GT")
# print(f"{'='*55}")

In [38]:
# # ── PRUNING CELL — run after manual review of ISIC 2020 images ────────────
# print("Syncing ISIC 2020 GT with files on disk ...")

# existing = {f.name for f in ISIC_DIR.glob("*.jpg")}
# gt_path  = GT_DIR / "dermatology_isic2020_gt.json"

# with open(gt_path) as f:
#     gt_all = json.load(f)

# gt_kept    = [r for r in gt_all if r["filename"] in existing]
# gt_removed = [r for r in gt_all if r["filename"] not in existing]

# with open(gt_path, "w") as f:
#     json.dump(gt_kept, f, indent=2)

# strata_final = {}
# for r in gt_kept:
#     strata_final[r["stratum"]] = strata_final.get(r["stratum"], 0) + 1

# print(f"\n{'='*50}")
# print(f"  Before : {len(gt_all)} records")
# print(f"  Removed: {len(gt_removed)}")
# print(f"  Kept   : {len(gt_kept)}")
# print(f"  Strata : {strata_final}")
# print(f"{'='*50}")

# target_total = sum(N["isic"].values())
# if len(gt_kept) < target_total:
#     print(f"\n  ⚠ {len(gt_kept)} images — need {target_total} for full eval")
# else:
#     print(f"\n  ✓ Ready for inference pipeline")

---
## Section 3 — Dermatology: HAM10000
**Source:** Kaggle — `kmader/skin-cancer-mnist-ham10000`  
**Auth:** Kaggle API key (see Section 0b)  
**Size:** ~2.4 GB  
**Images:** 10,015 dermoscopy images, 7 diagnostic categories  

**Requires:** `~/.kaggle/kaggle.json`


In [39]:
# # ── HAM10000 diagnosis → stratum ──────────────────────────────────────────
# HAM_STRATUM = {
#     "nv": "DS1", "df": "DS1", "bkl": "DS1", "vasc": "DS1",
#     "akiec": "DS2",
#     "mel": "DS3", "bcc": "DS3", "scc": "DS3",
# }
# HAM_FULL = {
#     "nv": "Melanocytic nevi", "df": "Dermatofibroma",
#     "bkl": "Benign keratosis", "vasc": "Vascular lesion",
#     "akiec": "Actinic keratosis / Bowen's disease",
#     "mel": "Melanoma", "bcc": "Basal cell carcinoma",
#     "scc": "Squamous cell carcinoma",
# }
# HAM_DIR = RAW_DIR / "dermatology" / "ham10000"
# HAM_DIR.mkdir(parents=True, exist_ok=True)
# HAM_TMP = TMP_DIR / "ham10000"
# HAM_TMP.mkdir(exist_ok=True)
# print(f"HAM10000 targets: {N['ham']}")

# # ── Kaggle download ────────────────────────────────────────────────────────
# import kagglehub

# print("Downloading HAM10000 from Kaggle (~2.4 GB) ...")
# path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
# print(f"✓ Download complete\nPath to dataset files: {path}")

# # ── Load metadata + sample ────────────────────────────────────────────────
# meta_paths = list(HAM_TMP.rglob("HAM10000_metadata.csv"))
# assert meta_paths, "metadata CSV not found — check download"

# df_ham = pd.read_csv(meta_paths[0])
# df_ham["stratum"]        = df_ham["dx"].map(HAM_STRATUM)
# df_ham["diagnosis_full"] = df_ham["dx"].map(HAM_FULL)
# df_ham = df_ham.dropna(subset=["stratum"])
# print(f"Loaded {len(df_ham)} records")
# print(f"Diagnosis distribution:\n{df_ham['dx'].value_counts()}")

# selected_ham = pd.concat([
#     df_ham[df_ham["stratum"] == s].sample(
#         n=min(n, len(df_ham[df_ham["stratum"] == s])),
#         random_state=RANDOM_SEED
#     )
#     for s, n in N["ham"].items()
# ])
# print(f"\nSelected {len(selected_ham)} images: {dict(selected_ham['stratum'].value_counts())}")

# # ── Find images + save ────────────────────────────────────────────────────
# def find_ham_image(img_id):
#     for p in HAM_TMP.rglob(f"{img_id}.*"):
#         if p.suffix.lower() in (".jpg", ".jpeg", ".png"):
#             return p
#     return None

# gt_ham    = []
# saved_ham = 0

# for _, row in tqdm(selected_ham.iterrows(), total=len(selected_ham), desc="HAM10000"):
#     img_path = find_ham_image(row["image_id"])
#     if img_path is None:
#         print(f"  Not found: {row['image_id']}"); continue

#     stratum = row["stratum"]
#     img_id  = f"ham10000_{stratum}_{saved_ham:04d}"
#     fname   = f"{img_id}.jpg"

#     Image.open(img_path).convert("RGB").save(str(HAM_DIR / fname), "JPEG", quality=95)

#     gt_ham.append({
#         "image_id":        img_id,
#         "filename":        fname,
#         "source":          "HAM10000",
#         "modality":        "dermatology",
#         "stratum":         stratum,
#         "original_id":     row["image_id"],
#         "diagnosis":       row["dx"],
#         "diagnosis_full":  row["diagnosis_full"],
#         "malignant":       stratum == "DS3",
#         "localization":    row.get("localization", ""),
#         "age":             row["age"] if pd.notna(row.get("age")) else None,
#         "sex":             row.get("sex", ""),
#         "dx_type":         row.get("dx_type", ""),
#         "expert_verified": row.get("dx_type") in ("histo", "follow_up", "consensus"),
#     })
#     saved_ham += 1

# gt_path = GT_DIR / "dermatology_ham10000_gt.json"
# with open(gt_path, "w") as f:
#     json.dump(gt_ham, f, indent=2)

# print(f"\n✓ HAM10000: {saved_ham} images → {HAM_DIR}")
# print(f"✓ Ground truth → {gt_path}")

---
## Section 4 — Histopathology: PatchCamelyon (PCam)
**Source:** HuggingFace — `zacharielegault/PatchCamelyon`  
**Auth:** None required  
**Size:** ~7.5 GB  
**Images:** 96×96 breast tissue patches (upsampled to 224×224 for VLM input)  
**Labels:** Binary — 0 = no tumour, 1 = tumour

**Stratification:**
- `HS1` — Non-tumour (label = 0)
- `HS3` — Tumour (label = 1)
- `HS2` — Approximated ambiguous (first N negative patches; PCam has no explicit ambiguous class)


In [40]:
# # ── Load + collect PCam patches ────────────────────────────────────────────
# from datasets import load_dataset as hf_load

# PCAM_DIR    = RAW_DIR / "histopathology" / "pcam"
# targets_pcam = N["pcam"]
# TARGET_SIZE  = 224   # upsample from 96px for VLM input

# print(f"Loading PatchCamelyon (zacharielegault/PatchCamelyon) ...")
# print(f"Targets: {targets_pcam}")

# ds_pcam = hf_load("1aurent/PatchCamelyon",
#                   split="train", streaming=True, trust_remote_code=True)

# n_hs1 = targets_pcam.get("HS1", 0)
# n_hs2 = targets_pcam.get("HS2", 0)
# n_hs3 = targets_pcam.get("HS3", 0)
# buckets_pcam = {"HS1": [], "HS2": [], "HS3": []}

# with tqdm(total=n_hs1+n_hs2+n_hs3, desc="PCam") as pbar:
#     for sample in ds_pcam:
#         label = sample.get("label", sample.get("Label", -1))
#         if hasattr(label, "item"): label  = label.item()
#         label = int(label)

#         if label == 1 and len(buckets_pcam["HS3"]) < n_hs3:
#             buckets_pcam["HS3"].append(sample); pbar.update(1)
#         elif label == 0:
#             if len(buckets_pcam["HS2"]) < n_hs2:
#                 buckets_pcam["HS2"].append(sample); pbar.update(1)
#             elif len(buckets_pcam["HS1"]) < n_hs1:
#                 buckets_pcam["HS1"].append(sample); pbar.update(1)

#         if all(len(buckets_pcam[s]) >= t for s,t in
#                [("HS1",n_hs1),("HS2",n_hs2),("HS3",n_hs3)]):
#             break

# print(f"Collected: { {s: len(b) for s,b in buckets_pcam.items()} }")


In [41]:
# # ── Save PCam patches + GT ─────────────────────────────────────────────────
# gt_pcam = []
# saved_pcam = 0

# for stratum, samples in buckets_pcam.items():
#     print(f"Saving {stratum}: {len(samples)} patches ...")
#     for idx, sample in enumerate(tqdm(samples, desc=f"  {stratum}", leave=False)):
#         img = sample.get("image", sample.get("Image"))
#         lbl = int(sample.get("label", sample.get("Label", 0)))
#         if hasattr(lbl, "item"): lbl = lbl.item()

#         if img.mode != "RGB": img = img.convert("RGB")
#         if img.size[0] < TARGET_SIZE:
#             img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.BICUBIC)

#         img_id = f"pcam_{stratum}_{idx:04d}"
#         fname  = f"{img_id}.jpg"
#         img.save(str(PCAM_DIR / fname), "JPEG", quality=95)

#         gt_pcam.append({
#             "image_id": img_id, "filename": fname,
#             "source": "PatchCamelyon", "modality": "histopathology",
#             "stratum": stratum, "tumor_present": bool(lbl == 1),
#             "original_label": lbl, "original_size": "96x96",
#             "saved_size": f"{TARGET_SIZE}x{TARGET_SIZE}",
#             "tissue_type": "breast", "expert_verified": True,
#             "ambiguous": stratum == "HS2",
#         })
#         saved_pcam += 1

# gt_path = GT_DIR / "histopathology_pcam_gt.json"
# with open(gt_path, "w") as f: json.dump(gt_pcam, f, indent=2)
# print(f"\n✓ PCam: {saved_pcam} patches → {PCAM_DIR}")
# print(f"✓ Ground truth → {gt_path}")
# print(f"  (All patches upsampled 96×96 → {TARGET_SIZE}×{TARGET_SIZE})")


---
## Section 5 — Histopathology: NCT-CRC-HE-100K
**Source:** HuggingFace — `1aurent/NCT-CRC-HE`  
**Auth:** None required  
**Size:** ~1.8 GB  
**Images:** 224×224 colorectal cancer H&E patches, 9 tissue classes  

**Stratification:**
| Class | Tissue | Stratum |
|---|---|---|
| NORM, STR, MUS, BACK | Normal / stroma | HS1 |
| LYM, DEB, ADI | Inflammatory / adipose | HS2 |
| TUM, MUC | Tumour epithelium / mucus | HS3 |


In [42]:
# # ── NCT tissue class mapping ───────────────────────────────────────────────
# NCT_STRATUM = {
#     "NORM":"HS1","STR":"HS1","MUS":"HS1","BACK":"HS1",
#     "LYM":"HS2","DEB":"HS2","ADI":"HS2",
#     "TUM":"HS3","MUC":"HS3",
# }
# NCT_FULL = {
#     "NORM":"Normal colon mucosa","STR":"Cancer-associated stroma",
#     "MUS":"Smooth muscle","BACK":"Background",
#     "LYM":"Lymphocytes","DEB":"Debris/mucus/blood","ADI":"Adipose tissue",
#     "TUM":"Colorectal adenocarcinoma","MUC":"Mucus",
# }
# # Integer label → class (HuggingFace integer encoding)
# NCT_INT_LABELS = ["ADI","BACK","DEB","LYM","MUC","MUS","NORM","STR","TUM"]

# NCT_DIR = RAW_DIR / "histopathology" / "nct_crc"
# n_hs1 = N["nct"].get("HS1", N["nct"].get("other", 50))
# n_hs2 = N["nct"].get("HS2", N["nct"].get("ambig", 20))
# n_hs3 = N["nct"].get("HS3", 50)
# targets_nct = {"HS1": n_hs1, "HS2": n_hs2, "HS3": n_hs3}
# print(f"NCT-CRC targets: {targets_nct}")


In [43]:
# # ── Load + collect NCT patches ─────────────────────────────────────────────
# print("Loading NCT-CRC-HE-100K (1aurent/NCT-CRC-HE) ...")
# ds_nct = hf_load("1aurent/NCT-CRC-HE",
#                  split="CRC_VAL_HE_7K", streaming=True, trust_remote_code=True)

# buckets_nct = {"HS1": [], "HS2": [], "HS3": []}

# with tqdm(total=sum(targets_nct.values()), desc="NCT-CRC") as pbar:
#     for sample in ds_nct:
#         label_raw = sample.get("label", sample.get("Label",""))
#         if isinstance(label_raw, int):
#             tissue = NCT_INT_LABELS[label_raw] if label_raw < len(NCT_INT_LABELS) else ""
#         else:
#             tissue = str(label_raw).upper()

#         stratum = NCT_STRATUM.get(tissue)
#         if stratum is None: continue
#         if len(buckets_nct[stratum]) >= targets_nct[stratum]: continue

#         buckets_nct[stratum].append({"sample": sample, "tissue": tissue})
#         pbar.update(1)
#         if all(len(buckets_nct[s]) >= targets_nct[s] for s in targets_nct):
#             break

# print(f"Collected: { {s: len(b) for s,b in buckets_nct.items()} }")


In [44]:
# # ── Save NCT patches + GT ──────────────────────────────────────────────────
# gt_nct = []
# saved_nct = 0

# for stratum, items in buckets_nct.items():
#     print(f"Saving {stratum}: {len(items)} patches ...")
#     for idx, item in enumerate(tqdm(items, desc=f"  {stratum}", leave=False)):
#         img    = item["sample"].get("image", item["sample"].get("Image"))
#         tissue = item["tissue"]

#         if img.mode != "RGB": img = img.convert("RGB")

#         img_id = f"nct_{stratum}_{idx:04d}"
#         fname  = f"{img_id}.jpg"
#         img.save(str(NCT_DIR / fname), "JPEG", quality=95)

#         gt_nct.append({
#             "image_id": img_id, "filename": fname,
#             "source": "NCT-CRC-HE-100K", "modality": "histopathology",
#             "stratum": stratum, "tissue_class": tissue,
#             "tissue_full": NCT_FULL.get(tissue, tissue),
#             "tumor_present": stratum == "HS3",
#             "ambiguous": stratum == "HS2",
#             "patch_size": "224x224", "stain": "H&E",
#             "expert_verified": True,
#         })
#         saved_nct += 1

# gt_path = GT_DIR / "histopathology_nct_gt.json"
# with open(gt_path, "w") as f: json.dump(gt_nct, f, indent=2)
# print(f"\n✓ NCT-CRC: {saved_nct} patches → {NCT_DIR}")
# print(f"✓ Ground truth → {gt_path}")


---
## Section 6 — Ophthalmology: REFUGE
**Source:** Kaggle — `andrewmvd/retinal-fundus-glaucoma`  
**Auth:** Kaggle API key  
**Size:** ~250 MB  
**Images:** 1,200 retinal fundus photographs with glaucoma labels + CDR values  

**Stratification:**
- `OS1` — Normal disc: glaucoma = 0, CDR < 0.5
- `OS2` — Borderline: CDR in [0.5, 0.7)
- `OS3` — Confirmed glaucoma: glaucoma = 1 or CDR ≥ 0.7


In [45]:
# # ── Helper: assign ophthalmology stratum ──────────────────────────────────
# def ophthal_stratum(glaucoma: int, cdr: float) -> str:
#     if glaucoma == 1 or cdr >= 0.7: return "OS3"
#     elif cdr >= 0.5:                 return "OS2"
#     else:                            return "OS1"

# REFUGE_DIR = RAW_DIR / "ophthalmology" / "refuge"
# REFUGE_TMP = TMP_DIR / "refuge"
# REFUGE_TMP.mkdir(exist_ok=True)
# print(f"REFUGE targets: {N['refuge']}")

In [46]:
# # ── Helper: assign ophthalmology stratum ──────────────────────────────────
# def ophthal_stratum(glaucoma: int, cdr: float) -> str:
#     if glaucoma == 1 or cdr >= 0.7: return "OS3"
#     elif cdr >= 0.5:                 return "OS2"
#     else:                            return "OS1"

# REFUGE_DIR = RAW_DIR / "ophthalmology" / "refuge"
# REFUGE_DIR.mkdir(parents=True, exist_ok=True)
# print(f"REFUGE targets: {N['refuge']}")

# # ── Kaggle download → use returned cache path directly ────────────────────
# import kagglehub
# from pathlib import Path

# print("Downloading REFUGE (andrewmvd/retinal-fundus-glaucoma) ...")
# cache_path = Path(kagglehub.dataset_download("andrewmvd/retinal-disease-classification"))
# print(f"✓ Download complete\nCache path: {cache_path}")
# print("Files found (first 20):")
# for p in sorted(cache_path.rglob("*"))[:20]:
#     print(f"  {p.relative_to(cache_path)}")

# # ── Load GT labels from cache ──────────────────────────────────────────────
# gt_labels_refuge = {}

# for csv_f in cache_path.rglob("*.csv"):
#     try:
#         df_r = pd.read_csv(csv_f)
#         print(f"CSV: {csv_f.name} | cols: {list(df_r.columns)[:6]}")
#         id_col  = next((c for c in df_r.columns if any(x in c.lower() for x in ["id","file","image","name"])), None)
#         glc_col = next((c for c in df_r.columns if any(x in c.lower() for x in ["label","glaucoma","diag"])), None)
#         cdr_col = next((c for c in df_r.columns if any(x in c.lower() for x in ["cdr","cup","ratio"])), None)
#         if id_col and glc_col:
#             for _, row in df_r.iterrows():
#                 k = str(row[id_col]).replace(".jpg","").replace(".png","").strip()
#                 gt_labels_refuge[k] = {
#                     "glaucoma": int(float(str(row[glc_col]).strip())),
#                     "cdr": float(row[cdr_col]) if cdr_col and pd.notna(row.get(cdr_col)) else 0.5,
#                 }
#             print(f"  → Loaded {len(gt_labels_refuge)} labels")
#     except Exception as e:
#         print(f"  Could not read {csv_f.name}: {e}")

# # ── Find + annotate + stratify images from cache ──────────────────────────
# img_files_refuge = [
#     f for f in (list(cache_path.rglob("*.jpg")) + list(cache_path.rglob("*.png")))
#     if "mask" not in str(f).lower() and "seg" not in str(f).lower()
# ]
# print(f"\nFound {len(img_files_refuge)} fundus images in cache")

# annotated_refuge = []
# for img_path in img_files_refuge:
#     img_id = img_path.stem
#     info   = gt_labels_refuge.get(img_id)
#     if info:
#         glaucoma, cdr = info["glaucoma"], info["cdr"]
#     else:
#         glaucoma = 1 if ("glaucoma" in str(img_path).lower() or img_id.startswith("g")) else 0
#         cdr      = 0.75 if glaucoma else 0.35
#     annotated_refuge.append({
#         "path": img_path, "img_id": img_id,
#         "glaucoma": glaucoma, "cdr": cdr,
#         "stratum": ophthal_stratum(glaucoma, cdr),
#     })

# # ── Stratified sampling ────────────────────────────────────────────────────
# import random
# random.seed(RANDOM_SEED)
# random.shuffle(annotated_refuge)

# buckets_refuge = {s: [] for s in N["refuge"]}
# for item in annotated_refuge:
#     s = item["stratum"]
#     if s in buckets_refuge and len(buckets_refuge[s]) < N["refuge"][s]:
#         buckets_refuge[s].append(item)

# print(f"Sampled: { {s: len(b) for s, b in buckets_refuge.items()} }")
# missing = {s: N['refuge'][s] - len(b) for s, b in buckets_refuge.items() if len(b) < N['refuge'][s]}
# if missing:
#     print(f"⚠ Shortfall vs targets: {missing}")

# # ── Save images + GT ───────────────────────────────────────────────────────
# gt_refuge    = []
# saved_refuge = 0

# for stratum, items in buckets_refuge.items():
#     print(f"Saving {stratum}: {len(items)} images ...")
#     for idx, item in enumerate(tqdm(items, desc=f"  {stratum}", leave=False)):
#         img_id = f"refuge_{stratum}_{idx:04d}"
#         fname  = f"{img_id}.jpg"
#         Image.open(item["path"]).convert("RGB").save(str(REFUGE_DIR / fname), "JPEG", quality=95)
#         gt_refuge.append({
#             "image_id":       img_id,
#             "filename":       fname,
#             "source":         "REFUGE",
#             "modality":       "ophthalmology",
#             "stratum":        stratum,
#             "original_id":    item["img_id"],
#             "glaucoma":       bool(item["glaucoma"]),
#             "cup_disc_ratio": round(item["cdr"], 3),
#             "cdr_suspicious": item["cdr"] >= 0.5,
#             "expert_verified": True,
#         })
#         saved_refuge += 1

# gt_path = GT_DIR / "ophthalmology_refuge_gt.json"
# with open(gt_path, "w") as f:
#     json.dump(gt_refuge, f, indent=2)

# print(f"\n✓ REFUGE: {saved_refuge} images → {REFUGE_DIR}")
# print(f"✓ Ground truth → {gt_path}")

---
## Section 7 — Ophthalmology: ORIGA-light
**Source:** Kaggle — `sshikamaru/glaucoma-detection`  
**Auth:** Kaggle API key  
**Images:** ~650 fundus images from Singapore Eye Research Institute  
**Note:** Original full ORIGA dataset is no longer publicly hosted. This Kaggle version contains the same images as ORIGA-light with identical ground truth labels.

Same stratification as REFUGE (OS1/OS2/OS3).


In [47]:
# # ── ORIGA download + process ───────────────────────────────────────────────
# import kagglehub
# import random
# from pathlib import Path

# ORIGA_DIR = RAW_DIR / "ophthalmology" / "origa"
# ORIGA_DIR.mkdir(parents=True, exist_ok=True)
# print(f"ORIGA targets: {N['origa']}")

# # ── Kaggle download → use returned cache path directly ────────────────────
# print("Downloading ORIGA (sshikamaru/glaucoma-detection) ...")
# cache_path = Path(kagglehub.dataset_download("sshikamaru/glaucoma-detection"))
# print(f"✓ Download complete\nCache path: {cache_path}")
# print("Files found (first 20):")
# for p in sorted(cache_path.rglob("*"))[:20]:
#     print(f"  {p.relative_to(cache_path)}")

# # ── Load GT labels from cache ──────────────────────────────────────────────
# gt_labels_origa = {}

# for csv_f in cache_path.rglob("*.csv"):
#     try:
#         df_o = pd.read_csv(csv_f)
#         print(f"CSV: {csv_f.name} | cols: {list(df_o.columns)[:6]}")
#         id_col  = next((c for c in df_o.columns if any(x in c.lower() for x in ["id","file","image","name"])), None)
#         glc_col = next((c for c in df_o.columns if any(x in c.lower() for x in ["label","glaucoma","diag"])), None)
#         cdr_col = next((c for c in df_o.columns if any(x in c.lower() for x in ["cdr","cup","ratio"])), None)
#         if id_col and glc_col:
#             for _, row in df_o.iterrows():
#                 k = str(row[id_col]).replace(".jpg","").replace(".png","").strip()
#                 gt_labels_origa[k] = {
#                     "glaucoma": int(float(str(row[glc_col]).strip())),
#                     "cdr": float(row[cdr_col]) if cdr_col and pd.notna(row.get(cdr_col)) else 0.5,
#                 }
#             print(f"  → Loaded {len(gt_labels_origa)} labels")
#     except Exception as e:
#         print(f"  Could not read {csv_f.name}: {e}")

# # ── Find + annotate images from cache ─────────────────────────────────────
# img_files_origa = [
#     f for f in (list(cache_path.rglob("*.jpg")) + list(cache_path.rglob("*.png")))
#     if "mask" not in str(f).lower() and "seg" not in str(f).lower()
# ]
# print(f"\nFound {len(img_files_origa)} fundus images in cache")

# annotated_origa = []
# for img_path in img_files_origa:
#     img_id = img_path.stem
#     info   = gt_labels_origa.get(img_id)
#     if info:
#         glaucoma, cdr = info["glaucoma"], info["cdr"]
#     else:
#         glaucoma = 1 if ("/1/" in str(img_path) or "glaucoma" in str(img_path).lower()) else 0
#         cdr      = 0.75 if glaucoma else 0.35
#     annotated_origa.append({
#         "path": img_path, "img_id": img_id,
#         "glaucoma": glaucoma, "cdr": cdr,
#         "stratum": ophthal_stratum(glaucoma, cdr),
#     })

# # ── Stratified sampling ────────────────────────────────────────────────────
# random.seed(RANDOM_SEED)
# random.shuffle(annotated_origa)

# buckets_origa = {s: [] for s in N["origa"]}
# for item in annotated_origa:
#     s = item["stratum"]
#     if s in buckets_origa and len(buckets_origa[s]) < N["origa"][s]:
#         buckets_origa[s].append(item)

# print(f"Sampled: { {s: len(b) for s, b in buckets_origa.items()} }")
# missing = {s: N['origa'][s] - len(b) for s, b in buckets_origa.items() if len(b) < N['origa'][s]}
# if missing:
#     print(f"⚠ Shortfall vs targets: {missing}")

# # ── Save images + GT ───────────────────────────────────────────────────────
# gt_origa    = []
# saved_origa = 0

# for stratum, items in buckets_origa.items():
#     print(f"Saving {stratum}: {len(items)} images ...")
#     for idx, item in enumerate(tqdm(items, desc=f"  {stratum}", leave=False)):
#         img_id = f"origa_{stratum}_{idx:04d}"
#         fname  = f"{img_id}.jpg"
#         Image.open(item["path"]).convert("RGB").save(str(ORIGA_DIR / fname), "JPEG", quality=95)
#         gt_origa.append({
#             "image_id":        img_id,
#             "filename":        fname,
#             "source":          "ORIGA-light",
#             "modality":        "ophthalmology",
#             "stratum":         stratum,
#             "original_id":     item["img_id"],
#             "glaucoma":        bool(item["glaucoma"]),
#             "cup_disc_ratio":  round(item["cdr"], 3),
#             "cdr_suspicious":  item["cdr"] >= 0.5,
#             "expert_verified": True,
#         })
#         saved_origa += 1

# gt_path = GT_DIR / "ophthalmology_origa_gt.json"
# with open(gt_path, "w") as f:
#     json.dump(gt_origa, f, indent=2)

# print(f"\n✓ ORIGA: {saved_origa} images → {ORIGA_DIR}")
# print(f"✓ Ground truth → {gt_path}")

---
## Section 8 — Merge & Validate Ground Truth
Combines per-dataset GT JSONs into one file per modality, verifies every image exists on disk and is readable, and reports the final stratum distribution.

**Run this after all 7 download sections complete.**


In [48]:
# ── GT file map ───────────────────────────────────────────────────────────
GT_MAP = {
    "radiology":      ["radiology_gt.json"],
    "dermatology":    ["dermatology_isic2020_gt.json", "dermatology_ham10000_gt.json"],
    "histopathology": ["histopathology_pcam_gt.json",  "histopathology_nct_gt.json"],
    "ophthalmology":  ["ophthalmology_refuge_gt.json", "ophthalmology_origa_gt.json"],
}

def load_gt_file(fname):
    p = GT_DIR / fname
    if not p.exists():
        print(f"  ⚠  Missing: {fname}"); return []
    with open(p) as f: return json.load(f)

def find_image_on_disk(rec):
    fname = rec.get("filename","")
    for p in (RAW_DIR).rglob(fname):
        return p
    return None

print("Ground truth files present:")
for modality, files in GT_MAP.items():
    for f in files:
        p = GT_DIR / f
        status = f"✓ ({len(load_gt_file(f))} records)" if p.exists() else "✗ MISSING"
        print(f"  {f:<45} {status}")


Ground truth files present:
  radiology_gt.json                             ✓ (2000 records)
  dermatology_isic2020_gt.json                  ✓ (2000 records)
  dermatology_ham10000_gt.json                  ✓ (1000 records)
  histopathology_pcam_gt.json                   ✓ (1400 records)
  histopathology_nct_gt.json                    ✓ (1000 records)
  ophthalmology_refuge_gt.json                  ✓ (700 records)
  ophthalmology_origa_gt.json                   ✓ (650 records)


In [49]:
# ── Validate + merge ──────────────────────────────────────────────────────
from collections import Counter

SUMMARY = {}

for modality, gt_files in GT_MAP.items():
    print(f"\n{'─'*50}")
    print(f"  {modality.upper()}")
    print(f"{'─'*50}")

    # Load all GT records for this modality
    all_records = []
    for fname in gt_files:
        recs = load_gt_file(fname)
        print(f"  {fname}: {len(recs)} records")
        all_records.extend(recs)

    # Validate each record: file exists, opens cleanly, min resolution
    valid, missing, corrupt = [], [], []
    for rec in tqdm(all_records, desc=f"  Validating {modality}", leave=False):
        p = find_image_on_disk(rec)
        if p is None:
            missing.append(rec["image_id"])
            continue
        try:
            img = Image.open(p)
            img.verify()                    # catches truncated / corrupt files
            img = Image.open(p)             # re-open after verify() exhausts the handle
            w, h = img.size
            if min(w, h) < 64:
                corrupt.append(f"{rec['image_id']} ({w}x{h} — too small)")
            else:
                rec["validated_path"] = str(p)
                rec["image_size"]     = f"{w}x{h}"
                valid.append(rec)
        except Exception as e:
            corrupt.append(f"{rec['image_id']} ({e})")

    # Report
    print(f"  ✓ Valid:   {len(valid):>5}")
    print(f"  ✗ Missing: {len(missing):>5}" + (f"  — first 3: {missing[:3]}" if missing else ""))
    print(f"  ✗ Corrupt: {len(corrupt):>5}" + (f"  — first 3: {corrupt[:3]}" if corrupt else ""))

    strata  = dict(Counter(r.get("stratum") for r in valid))
    sources = dict(Counter(r.get("source")  for r in valid))
    print(f"  Strata:  {strata}")
    print(f"  Sources: {sources}")

    # Merge + save
    merged_path = GT_DIR / f"{modality}_gt_merged.json"
    with open(merged_path, "w") as f:
        json.dump(valid, f, indent=2)
    print(f"  → Saved: {merged_path.name}  ({merged_path.stat().st_size / 1024:.1f} KB)")

    SUMMARY[modality] = {
        "n":       len(valid),
        "strata":  strata,
        "sources": sources,
        "missing": len(missing),
        "corrupt": len(corrupt),
    }

# ── Print final summary ────────────────────────────────────────────────────
print(f"\n{'═'*50}")
print("  FINAL SUMMARY")
print(f"{'═'*50}")
total = 0
for modality, s in SUMMARY.items():
    print(f"  {modality:<20} n={s['n']:>5}  missing={s['missing']}  corrupt={s['corrupt']}")
    total += s["n"]
print(f"{'─'*50}")
print(f"  {'TOTAL':<20} n={total:>5}")


──────────────────────────────────────────────────
  RADIOLOGY
──────────────────────────────────────────────────
  radiology_gt.json: 2000 records


  Validating radiology:   0%|          | 0/2000 [00:00<?, ?it/s]

  ✓ Valid:    2000
  ✗ Missing:     0
  ✗ Corrupt:     0
  Strata:  {'S1': 500, 'S2': 500, 'S3': 500, 'S4': 500}
  Sources: {'CheXpert-v1.0': 2000}
  → Saved: radiology_gt_merged.json  (1303.1 KB)

──────────────────────────────────────────────────
  DERMATOLOGY
──────────────────────────────────────────────────
  dermatology_isic2020_gt.json: 2000 records
  dermatology_ham10000_gt.json: 1000 records


  Validating dermatology:   0%|          | 0/3000 [00:00<?, ?it/s]

  ✓ Valid:    3000
  ✗ Missing:     0
  ✗ Corrupt:     0
  Strata:  {'DS1': 1633, 'DS2': 383, 'DS3': 984}
  Sources: {'ISIC2020': 2000, 'HAM10000': 1000}
  → Saved: dermatology_gt_merged.json  (1547.9 KB)

──────────────────────────────────────────────────
  HISTOPATHOLOGY
──────────────────────────────────────────────────
  histopathology_pcam_gt.json: 1400 records
  histopathology_nct_gt.json: 1000 records


  Validating histopathology:   0%|          | 0/2400 [00:00<?, ?it/s]

  ✓ Valid:    2400
  ✗ Missing:     0
  ✗ Corrupt:     0
  Strata:  {'HS1': 1100, 'HS3': 1100, 'HS2': 200}
  Sources: {'PatchCamelyon': 1400, 'NCT-CRC-HE-100K': 1000}
  → Saved: histopathology_gt_merged.json  (1172.2 KB)

──────────────────────────────────────────────────
  OPHTHALMOLOGY
──────────────────────────────────────────────────
  ophthalmology_refuge_gt.json: 700 records
  ophthalmology_origa_gt.json: 650 records


  Validating ophthalmology:   0%|          | 0/1350 [00:00<?, ?it/s]

  ✓ Valid:    1350
  ✗ Missing:     0
  ✗ Corrupt:     0
  Strata:  {'OS1': 1000, 'OS2': 200, 'OS3': 150}
  Sources: {'REFUGE': 700, 'ORIGA-light': 650}
  → Saved: ophthalmology_gt_merged.json  (576.3 KB)

══════════════════════════════════════════════════
  FINAL SUMMARY
══════════════════════════════════════════════════
  radiology            n= 2000  missing=0  corrupt=0
  dermatology          n= 3000  missing=0  corrupt=0
  histopathology       n= 2400  missing=0  corrupt=0
  ophthalmology        n= 1350  missing=0  corrupt=0
──────────────────────────────────────────────────
  TOTAL                n= 8750


In [50]:
# ── Final summary table ────────────────────────────────────────────────────
print("\n" + "="*65)
print("  MEDVIHALL DATASET SUMMARY")
print("="*65)
total = 0
for modality, info in SUMMARY.items():
    flag = "✓" if info["missing"] == 0 and info["corrupt"] == 0 else "⚠"
    print(f"  {flag} {modality:<20} {info['n']:>5} images")
    print(f"    Strata:  {info['strata']}")
    print(f"    Sources: {info['sources']}")
    if info["missing"] or info["corrupt"]:
        print(f"    Issues:  {info['missing']} missing, {info['corrupt']} corrupt")
    total += info["n"]
print(f"{'─'*65}")
print(f"  {'TOTAL':<20} {total:>5} images")
print("="*65)

if total < 400:
    print("\n⚠  WARNING: Fewer than expected images. Re-run failed sections.")
else:
    print("\n✓ Dataset ready.")
    print("  Merged GT files:")
    for f in sorted(GT_DIR.glob("*_merged.json")):
        n = len(json.load(open(f)))
        print(f"    {f.name}  ({n} records)")
    print("\n  Next step: run inference pipeline on these images.")



  MEDVIHALL DATASET SUMMARY
  ✓ radiology             2000 images
    Strata:  {'S1': 500, 'S2': 500, 'S3': 500, 'S4': 500}
    Sources: {'CheXpert-v1.0': 2000}
  ✓ dermatology           3000 images
    Strata:  {'DS1': 1633, 'DS2': 383, 'DS3': 984}
    Sources: {'ISIC2020': 2000, 'HAM10000': 1000}
  ✓ histopathology        2400 images
    Strata:  {'HS1': 1100, 'HS3': 1100, 'HS2': 200}
    Sources: {'PatchCamelyon': 1400, 'NCT-CRC-HE-100K': 1000}
  ✓ ophthalmology         1350 images
    Strata:  {'OS1': 1000, 'OS2': 200, 'OS3': 150}
    Sources: {'REFUGE': 700, 'ORIGA-light': 650}
─────────────────────────────────────────────────────────────────
  TOTAL                 8750 images

✓ Dataset ready.
  Merged GT files:
    dermatology_gt_merged.json  (3000 records)
    histopathology_gt_merged.json  (2400 records)
    ophthalmology_gt_merged.json  (1350 records)
    radiology_gt_merged.json  (2000 records)

  Next step: run inference pipeline on these images.


In [51]:
# # ═══════════════════════════════════════════════════════════════════════════
# # OPHTHALMOLOGY TOP-UP
# # Current : OS1=1000, OS2=200, OS3=150
# # Target  : OS1=1000, OS2=500, OS3=500
# # Shortfall: OS2 +300, OS3 +350
# # Sources :
# #   1. EyePACS AIROGS-light-v2  (large screening dataset, binary labels)
# #   2. G1020                    (1020 fundus images, expert CDR + glaucoma)
# # ═══════════════════════════════════════════════════════════════════════════

# import json, random
# import kagglehub
# import pandas as pd
# from pathlib import Path
# from PIL import Image
# from tqdm.notebook import tqdm

# OPTHAL_DIR = RAW_DIR / "ophthalmology"
# OPTHAL_DIR.mkdir(exist_ok=True)
# gt_path    = GT_DIR / "ophthalmology_gt_merged.json"

# # ── Load current state ─────────────────────────────────────────────────────
# with open(gt_path) as f:
#     gt_existing = json.load(f)

# current = {}
# for r in gt_existing:
#     current[r["stratum"]] = current.get(r["stratum"], 0) + 1
# print(f"Current state  : {current}")

# FINAL_TARGETS = {"OS1": 1000, "OS2": 500, "OS3": 500}
# needed = {s: max(0, FINAL_TARGETS[s] - current.get(s, 0))
#           for s in FINAL_TARGETS}
# print(f"Still needed   : {needed}")
# print(f"Total to add   : {sum(needed.values())} images\n")

# def ophthal_stratum(glaucoma: int, cdr: float) -> str:
#     if glaucoma == 1 or cdr >= 0.7: return "OS3"
#     elif cdr >= 0.5:                 return "OS2"
#     else:                            return "OS1"

# random.seed(RANDOM_SEED)
# new_gt  = []
# n_added = 0

# # ══════════════════════════════════════════════════════════════════════════
# # SOURCE 1 — EyePACS AIROGS-light-v2
# # Binary labels: NRG=no referable glaucoma (→OS2), RG=referable (→OS3)
# # ══════════════════════════════════════════════════════════════════════════
# print("="*55)
# print("Source 1 — EyePACS AIROGS-light-v2")
# print("="*55)
# print("Downloading via kagglehub (~1.5 GB) ...")

# airogs_path = Path(kagglehub.dataset_download(
#     "deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2"
# ))
# print(f"✓ Path: {airogs_path}")
# print("\nContents:")
# for p in sorted(airogs_path.iterdir()):
#     tag = f"({len(list(p.glob('*')))} files)" if p.is_dir() else f"({p.stat().st_size/1e6:.1f} MB)"
#     print(f"  {'📁' if p.is_dir() else '📄'} {p.name}  {tag}")

# # Load labels
# csv_candidates = list(airogs_path.rglob("*.csv"))
# print(f"\nCSVs: {[c.name for c in csv_candidates]}")

# df_air = pd.read_csv(csv_candidates[0])
# print(f"Loaded {len(df_air):,} records | columns: {list(df_air.columns)}")

# # Detect columns
# id_col  = next((c for c in df_air.columns
#                 if any(x in c.lower() for x in ["image","file","id","name"])),
#                df_air.columns[0])
# lbl_col = next((c for c in df_air.columns
#                 if any(x in c.lower() for x in ["label","class","target","glaucoma","diagnosis"])),
#                df_air.columns[-1])
# print(f"Using: id='{id_col}', label='{lbl_col}'")
# print(f"Label distribution:\n{df_air[lbl_col].value_counts().to_string()}")

# # Map labels → strata
# # AIROGS: "RG" or 1 = referable glaucoma → OS3
# #         "NRG" or 0 = no referable glaucoma → OS2 (borderline screening)
# def airogs_to_stratum(val) -> str:
#     v = str(val).strip().upper()
#     if v in ("RG", "1", "1.0", "GLAUCOMA", "POSITIVE"):
#         return "OS3"
#     return "OS2"

# df_air["stratum"] = df_air[lbl_col].apply(airogs_to_stratum)
# print(f"\nAIROGS stratum distribution: {dict(df_air['stratum'].value_counts())}")

# # Build image index
# print("\nIndexing images ...")
# airogs_img_index = {}
# for ext in ("*.jpg","*.jpeg","*.png","*.JPG","*.PNG"):
#     for p in airogs_path.rglob(ext):
#         airogs_img_index[p.stem] = p
# print(f"Indexed {len(airogs_img_index):,} images")

# # Sample + save — only what we still need
# airogs_topup_dir = OPTHAL_DIR / "airogs"
# airogs_topup_dir.mkdir(exist_ok=True)

# for stratum in ["OS2", "OS3"]:
#     n_needed = needed.get(stratum, 0)
#     if n_needed == 0:
#         print(f"\n  {stratum}: already at target, skipping")
#         continue

#     pool     = df_air[df_air["stratum"] == stratum]
#     n_sample = min(int(n_needed * 1.15), len(pool))   # 15% buffer
#     sampled  = pool.sample(n=n_sample, random_state=RANDOM_SEED)
#     saved_s  = 0

#     print(f"\n  {stratum}: need {n_needed}, sampling {n_sample} from {len(pool):,} available ...")

#     for _, row in tqdm(sampled.iterrows(), total=len(sampled),
#                        desc=f"  AIROGS {stratum}", leave=False):
#         if saved_s >= n_needed:
#             break

#         img_stem = str(row[id_col]).replace(".jpg","").replace(".png","").strip()
#         src      = airogs_img_index.get(img_stem)
#         if src is None:
#             continue

#         img_id = f"airogs_{stratum}_{n_added:04d}"
#         fname  = f"{img_id}.jpg"
#         fpath  = airogs_topup_dir / fname

#         if not fpath.exists():
#             try:
#                 Image.open(src).convert("RGB").save(str(fpath), "JPEG", quality=95)
#             except Exception as e:
#                 print(f"    Failed {img_stem}: {e}")
#                 continue

#         new_gt.append({
#             "image_id":        img_id,
#             "filename":        fname,
#             "source":          "EyePACS-AIROGS",
#             "modality":        "ophthalmology",
#             "stratum":         stratum,
#             "original_id":     img_stem,
#             "glaucoma":        stratum == "OS3",
#             "cup_disc_ratio":  None,
#             "cdr_suspicious":  True,
#             "expert_verified": True,
#         })
#         saved_s += 1
#         n_added += 1

#     # Update needed count for Source 2 fallback
#     needed[stratum] = max(0, needed[stratum] - saved_s)
#     print(f"  ✓ Added {saved_s} from AIROGS | still needed: {needed[stratum]}")

# # ══════════════════════════════════════════════════════════════════════════
# # SOURCE 2 — G1020 (fallback if AIROGS didn't fill OS2/OS3 fully)
# # 1,020 fundus images with expert CDR measurements + glaucoma labels
# # ══════════════════════════════════════════════════════════════════════════
# still_needed = sum(needed[s] for s in ["OS2","OS3"])

# if still_needed > 0:
#     print(f"\n{'='*55}")
#     print(f"Source 2 — G1020 (need {still_needed} more images)")
#     print(f"{'='*55}")
#     print("Downloading via kagglehub ...")

#     try:
#         g1020_path = Path(kagglehub.dataset_download("arnavjain1/glaucoma-datasets"))
#         print(f"✓ Path: {g1020_path}")
#         for p in sorted(g1020_path.iterdir()):
#             tag = f"({len(list(p.glob('*')))} files)" if p.is_dir() else f"({p.stat().st_size/1e6:.1f} MB)"
#             print(f"  {'📁' if p.is_dir() else '📄'} {p.name}  {tag}")

#         # Find CSVs
#         g_csvs = list(g1020_path.rglob("*.csv"))
#         print(f"\nCSVs: {[c.name for c in g_csvs]}")

#         if g_csvs:
#             df_g = pd.read_csv(g_csvs[0])
#             print(f"Loaded {len(df_g)} records | columns: {list(df_g.columns)}")

#             g_id_col  = next((c for c in df_g.columns
#                               if any(x in c.lower() for x in ["image","file","id","name"])),
#                              df_g.columns[0])
#             g_lbl_col = next((c for c in df_g.columns
#                               if any(x in c.lower() for x in ["label","glaucoma","class","target"])),
#                              None)
#             g_cdr_col = next((c for c in df_g.columns
#                               if any(x in c.lower() for x in ["cdr","cup","ratio"])), None)

#             print(f"Using: id='{g_id_col}', label='{g_lbl_col}', cdr='{g_cdr_col}'")

#             # Build image index
#             g_img_index = {}
#             for ext in ("*.jpg","*.jpeg","*.png","*.JPG","*.PNG"):
#                 for p in g1020_path.rglob(ext):
#                     g_img_index[p.stem] = p
#             print(f"Indexed {len(g_img_index):,} images")

#             g1020_topup_dir = OPTHAL_DIR / "g1020"
#             g1020_topup_dir.mkdir(exist_ok=True)

#             for stratum in ["OS2","OS3"]:
#                 n_needed_g = needed.get(stratum, 0)
#                 if n_needed_g == 0:
#                     continue

#                 # Filter by label
#                 if g_lbl_col:
#                     if stratum == "OS3":
#                         pool_g = df_g[df_g[g_lbl_col].astype(str).str.strip().isin(
#                             ["1","1.0","glaucoma","positive","RG"])]
#                     else:
#                         pool_g = df_g[df_g[g_lbl_col].astype(str).str.strip().isin(
#                             ["0","0.0","normal","negative","NRG"])]
#                 else:
#                     pool_g = df_g

#                 sampled_g = pool_g.sample(
#                     n=min(n_needed_g + 20, len(pool_g)),
#                     random_state=RANDOM_SEED
#                 )
#                 saved_g = 0

#                 print(f"\n  {stratum}: adding {n_needed_g} from G1020 ...")
#                 for _, row in tqdm(sampled_g.iterrows(), total=len(sampled_g),
#                                    desc=f"  G1020 {stratum}", leave=False):
#                     if saved_g >= n_needed_g:
#                         break

#                     img_stem = str(row[g_id_col]).replace(".jpg","").replace(".png","").strip()
#                     src      = g_img_index.get(img_stem)
#                     if src is None:
#                         continue

#                     cdr = float(row[g_cdr_col]) if g_cdr_col and pd.notna(row.get(g_cdr_col)) else (
#                         0.75 if stratum == "OS3" else 0.55
#                     )
#                     glaucoma = 1 if stratum == "OS3" else 0

#                     img_id = f"g1020_{stratum}_{n_added:04d}"
#                     fname  = f"{img_id}.jpg"
#                     fpath  = g1020_topup_dir / fname

#                     if not fpath.exists():
#                         try:
#                             Image.open(src).convert("RGB").save(str(fpath), "JPEG", quality=95)
#                         except Exception as e:
#                             continue

#                     new_gt.append({
#                         "image_id":        img_id,
#                         "filename":        fname,
#                         "source":          "G1020",
#                         "modality":        "ophthalmology",
#                         "stratum":         stratum,
#                         "original_id":     img_stem,
#                         "glaucoma":        bool(glaucoma),
#                         "cup_disc_ratio":  round(cdr, 3),
#                         "cdr_suspicious":  cdr >= 0.5,
#                         "expert_verified": True,
#                     })
#                     saved_g += 1
#                     n_added += 1

#                 print(f"  ✓ Added {saved_g} from G1020")

#     except Exception as e:
#         print(f"  G1020 download failed: {e}")
#         print("  Proceeding with AIROGS images only")

# # ══════════════════════════════════════════════════════════════════════════
# # Merge + save updated GT
# # ══════════════════════════════════════════════════════════════════════════
# gt_combined = gt_existing + new_gt

# with open(gt_path, "w") as f:
#     json.dump(gt_combined, f, indent=2)

# # Final counts
# final_strata = {}
# final_sources = {}
# for r in gt_combined:
#     final_strata[r["stratum"]]   = final_strata.get(r["stratum"], 0) + 1
#     final_sources[r["source"]]   = final_sources.get(r["source"], 0) + 1

# print(f"\n{'='*55}")
# print(f"✓ Ophthalmology top-up complete")
# print(f"  Added    : {n_added} images")
# print(f"  Total    : {len(gt_combined)} images")
# print(f"  Strata   : {final_strata}")
# print(f"  Sources  : {final_sources}")
# print(f"  GT saved : {gt_path}")
# print(f"{'='*55}")

# # Check against targets
# print("\nTarget check:")
# for s, target in FINAL_TARGETS.items():
#     have   = final_strata.get(s, 0)
#     status = "✓" if have >= target else f"⚠ SHORT by {target - have}"
#     print(f"  {s}: {have} / {target}  {status}")